In [0]:
%pip install pdfplumber

In [0]:
import requests
import os 
import pdfplumber
import time
from pathlib import Path
from datetime import datetime

In [0]:
url = "https://lacity.primegov.com/api/v2/PublicPortal/ListArchivedMeetingsByDays?days=3650"
response= requests.get(url)

if response.status_code == 200:
    meetings = response.json()
    #print(meetings)
else:
    print(f"Request failed with status code {response.status_code}")

filtered_for_plum = [m for m in meetings if m.get('committeeId') == 12]
print(filtered_for_plum)


results = []
for m in filtered_for_plum:
    for d in m['documentList']:
        if 'Agenda' in d['templateName'] and d['compileOutputType'] == 1:
            results.append({
                "meeting_id":m['id'],
                "meeting_date":m['date'],
                "meeting_title":m['title'],
                "document_id":d['id'],
                "templateId":d['templateId'],
                "templateName":d['templateName'],
                })
print(results)
        

#Writting The PDFs To The Bronze Volume

In [0]:
for r in results:
    file_path = f"/Volumes/la_lakehouse/bronze/plu_pdfs/{r['document_id']}_{r['meeting_id']}.pdf"
    if os.path.exists(file_path):
        print(f"File already exists for {r['document_id']}_{r['meeting_id']}.pdf")
        continue
    
    try:
        response = requests.get(url, timeout=15)
        time.sleep(1)
        if response.status_code == 200:
            print(f"Request successful for {r['document_id']}_{r['meeting_id']}")
            with open(file_path, "wb") as f:
                f.write(response.content)
        else:
            print(f"Request failed with status code {response.status_code} for {r['meeting_id']}")
    except requests.exceptions.RequestException as e:
        print(f"Request error for {r['meeting_id']}: {e}")
        continue

In [0]:
print(len(os.listdir("/Volumes/la_lakehouse/bronze/plu_pdfs/")))
print(len(results))

#Extracting Text From PDFs

In [0]:
pdf_dir = Path("/Volumes/la_lakehouse/bronze/plu_pdfs/")
pdf_list = []
for r in results:
    pdf_path = pdf_dir / f"{r['document_id']}_{r['meeting_id']}.pdf"
    pdf_text = ""
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                text = page.extract_text()
                if text:
                    pdf_text += text + "\n"
            pdf_dict = {
                "document_id":r["document_id"],
                "meeting_id":r["meeting_id"],
                "meeting_date":r["meeting_date"],
                "meeting_title":r["meeting_title"],
                "templateId":r["templateId"],
                "templateName":r["templateName"],
                "raw_text":pdf_text,
                "extraction_failed":False,
                "ingested_at": datetime.now(),
            }
            pdf_list.append(pdf_dict)
    except Exception as e:
        print(f"Error processing {pdf_path}: {e}")
        pdf_dict = {
                "document_id":r["document_id"],
                "meeting_id":r["meeting_id"],
                "meeting_date":r["meeting_date"],
                "meeting_title":r["meeting_title"],
                "templateId":r["templateId"],
                "templateName":r["templateName"],
                "raw_text":"",
                "extraction_failed":True,
                "ingested_at": datetime.now(),
            }
        pdf_list.append(pdf_dict)

In [0]:
print(len(pdf_list))
print(sum(1 for p in pdf_list if p['extraction_failed']))

# Converting Texts To A DataFrame And Write to Bronze Catalog

In [0]:
df = spark.createDataFrame(pdf_list)
df.write.mode("overwrite").format('delta').saveAsTable("la_lakehouse.bronze.plu_agendas")

#Testing Table

In [0]:
%sql
SELECT COUNT(*)
FROM la_lakehouse.bronze.plu_agendas

In [0]:
%sql
SELECT raw_text
FROM la_lakehouse.bronze.plu_agendas
WHERE document_id = 84459

In [0]:
%sql
SELECT COUNT(*)
FROM la_lakehouse.bronze.plu_agendas
WHERE raw_text = ''

#